# Land Cleared by Humanitarian Demining in Colombia, Before and After the 2016 Peace Agreement

**Author:** Travis Carlson
**Dataset:** *Situación de Desminado Humanitario en Colombia*, published by the Departamento Administrativo de la Presidencia de la República (DAPRE) on Colombia's national open data portal, datos.gov.co (CC BY-SA 4.0)

This notebook analyses Colombia's official record of humanitarian demining results from 2004 to 2023: 5,721 records of land cleared, explosive items found, surveys and hazardous areas, by municipality, month and operator. It checks and describes the data, compares three visual models and runs one hypothesis test on a question set in advance.

**Question.** Did humanitarian demining in Colombia clear more land each month after the 2016 peace agreement than before it?

**Analysis plan, fixed before the test was run.** Initial data analysis examines a dataset's properties for a research question set in advance and refrains from analyses related to that question, because decisions made after looking at the data can bias the results (Lusa et al., 2024). The plan:
- **Unit:** the national total of land cleared in each month: 84 months from January 2010 to December 2016 against 84 months from January 2017 to December 2023. A month with no record counts as zero.
- **Test:** Mann–Whitney U, two-sided, with α = 0.05. Reported: U, the p-value, the median of each period and the rank-biserial correlation as the effect size.
- **Check:** the same test on yearly totals, 7 years against 7, which does not treat months as independent.
- **Left out of the test:** 2004 to 2009, while operations were starting (1 to 50 records a year). These years still appear in the descriptive statistics and the first chart.
- The yearly totals had been seen when the question was chosen, so the test confirms a visible pattern rather than a blind prediction.

Sections 2 and 3 check the data and screen the properties the test depends on; sections 4 and 5 compare land cleared between the periods.

*Reference:* Lusa, L., Proust-Lima, C., Schmidt, C. O., Lee, K. J., le Cessie, S., Baillie, M., Lawrence, F., & Huebner, M. (2024). Initial data analysis for longitudinal studies to build a solid foundation for reproducible analysis. *PLOS ONE, 19*(5), e0295726. https://doi.org/10.1371/journal.pone.0295726

## 1. Setup

Libraries, file locations, the provenance of the raw file, the fixed settings of the analysis plan, and one chart style used throughout. The analysis uses Python, NumPy, Pandas, Matplotlib, Seaborn and SciPy.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import scipy
from scipy import stats

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

# Files (the raw file sits in the same folder as this notebook)
DATA_FILE = Path("desminado_humanitario.csv")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Provenance of the raw file
SOURCE_PAGE = ("https://www.datos.gov.co/Inclusi-n-Social-y-Reconciliaci-n/"
               "Situaci-n-de-Desminado-Humanitario-en-Colombia/jbqm-s5tn")
DOWNLOAD_URL = "https://www.datos.gov.co/api/views/jbqm-s5tn/rows.csv?accessType=DOWNLOAD"
DOWNLOADED_ON = "2026-10-18"
EXPECTED_SHA256 = "eda568a71903f05d7468c582b3795dbd5a537a71c5413f84f72f0acbcffb6617"

# Analysis plan, fixed before the test was run (see the introduction)
START_UP, BEFORE, AFTER = "2004–2009", "2010–2016", "2017–2023"
ALPHA = 0.05
MILITARY_OPERATORS = {"BRDEH", "BDIAN"}  # the Army's and the Navy's humanitarian demining units

# Chart style: light surface, quiet grid, and one fixed colour per period in every chart
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"
PERIOD_COLOURS = {START_UP: MUTED, BEFORE: "#2a78d6", AFTER: "#eb6834"}
THOUSANDS = FuncFormatter(lambda value, _: f"{value:,.0f}")

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 1, "grid.linestyle": "-",
    "axes.axisbelow": True, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "font.size": 10, "lines.linewidth": 2, "legend.frameon": False,
})

for name, module in [("pandas", pd), ("numpy", np), ("matplotlib", matplotlib),
                     ("seaborn", sns), ("scipy", scipy)]:
    print(f"{name:<10} {module.__version__}")

pandas     3.0.6
numpy      2.5.4
matplotlib 3.11.2
seaborn    0.13.2
scipy      1.18.1


## 2. Data handling

### 2.1 Load the raw file

The raw file is a snapshot of the portal's CSV export, kept unchanged next to this notebook. Its SHA-256 checksum is compared with the one recorded when it was downloaded, so anyone rerunning the notebook can confirm they are using the same data. The portal's metadata gives the publisher, the licence, a description of each column and the date the rows were last updated (19 January 2024).

In [2]:
def file_sha256(path):
    """Return the SHA-256 checksum of a file, reading it in 1 MB chunks.

    Used to confirm that the raw file is the snapshot recorded when it was downloaded.
    A different checksum means a newer download or a changed file.
    """
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


checksum = file_sha256(DATA_FILE)
print(f"Source page:   {SOURCE_PAGE}")
print(f"Downloaded on: {DOWNLOADED_ON}")
print(f"SHA-256:       {checksum}")
if checksum == EXPECTED_SHA256:
    print("Checksum matches the recorded snapshot.")
else:
    print("WARNING: checksum differs from the recorded snapshot; results may not match the report.")

df_raw = pd.read_csv(DATA_FILE, encoding="utf-8")
print(f"\nLoaded {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

Source page:   https://www.datos.gov.co/Inclusi-n-Social-y-Reconciliaci-n/Situaci-n-de-Desminado-Humanitario-en-Colombia/jbqm-s5tn
Downloaded on: 2026-10-18
SHA-256:       eda568a71903f05d7468c582b3795dbd5a537a71c5413f84f72f0acbcffb6617
Checksum matches the recorded snapshot.

Loaded 5,721 rows x 18 columns


,departamento,codigodanedepartamento,municipio,codigodanemunicipio,ano,mes,tipodesminado,areadespejada,map,muse,aei,totalartefactosdestruidos,organizacion,estudiosnotecnicos,estudiostecnicos,operacionesdespeje,areaspeligrosas,areaspeligrosasconfirmadas
0,CAUCA,19,CALOTO,19142,2018,4,Comunidades,0.0,0,0,0,0,HI,3,0,0,0,0
1,META,50,LA MACARENA,50350,2019,9,Comunidades,1563.0,0,148,0,148,BRDEH,0,0,1,0,0
2,SANTANDER,68,SIMACOTA,68745,2020,7,Comunidades,6499.0,0,0,0,0,BRDEH,0,0,1,0,0
3,TOLIMA,73,RONCESVALLES,73622,2022,10,Comunidades,0.0,0,0,0,0,BRDEH,0,0,0,1,0
4,CAUCA,19,PÁEZ,19517,2023,2,Comunidades,0.0,0,0,0,0,HI,4,0,0,0,0


In [3]:
df_raw.info()
fractional = (df_raw["areadespejada"] % 1 != 0).sum()
print(f"\nRecords whose land cleared has a fractional part: {fractional}")

<class 'pandas.DataFrame'>
RangeIndex: 5721 entries, 0 to 5720
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   departamento                5721 non-null   str    
 1   codigodanedepartamento      5721 non-null   int64  
 2   municipio                   5721 non-null   str    
 3   codigodanemunicipio         5721 non-null   int64  
 4   ano                         5721 non-null   int64  
 5   mes                         5721 non-null   int64  
 6   tipodesminado               5721 non-null   str    
 7   areadespejada               5721 non-null   float64
 8   map                         5721 non-null   int64  
 9   muse                        5721 non-null   int64  
 10  aei                         5721 non-null   int64  
 11  totalartefactosdestruidos   5721 non-null   int64  
 12  organizacion                5721 non-null   str    
 13  estudiosnotecnicos          5721 non-null   

All 18 columns loaded, with no empty cells. Text columns were read as strings and the counts as integers. Two types need attention: the official DANE codes for departments and municipalities were read as numbers, which drops their leading zero, and land cleared was read as a decimal number because 67 records report fractions of a square metre.

### 2.2 Data dictionary and preparation

Column descriptions from the portal's metadata, with the English names used below.

| Published column | English name | Meaning (portal description) |
|---|---|---|
| `departamento`, `codigodanedepartamento` | `department`, `department_code` | Department where demining took place, and its DANE code |
| `municipio`, `codigodanemunicipio` | `municipality`, `municipality_code` | Municipality, and its DANE code |
| `ano`, `mes` | `year`, `month` | Year and month of the work |
| `tipodesminado` | `demining_type` | Demining on military bases or in communities |
| `areadespejada` | `area_cleared_m2` | Land cleared, in square metres |
| `map`, `muse`, `aei` | `items_mines`, `items_uxo`, `items_ieds` | Items found: antipersonnel mines, unexploded ordnance and improvised explosive devices |
| `totalartefactosdestruidos` | `items_total` | Total items. The column name says destroyed; the portal's description says found |
| `organizacion` | `operator` | Organisation that reported the information |
| `estudiosnotecnicos`, `estudiostecnicos` | `non_technical_surveys`, `technical_surveys` | Non-technical and technical surveys carried out |
| `operacionesdespeje` | `clearance_operations` | Clearance operations carried out |
| `areaspeligrosas`, `areaspeligrosasconfirmadas` | `hazardous_areas`, `confirmed_hazardous_areas` | Hazardous areas recorded, and confirmed |

The preparation function renames the columns, stores the codes as zero-padded text (the reliable key, since some municipality names repeat across departments), translates the demining type, and adds two grouping columns: the operator type (military or civilian) and the period of the analysis plan. It returns a new table, so the raw data stays unchanged.

In [4]:
COLUMN_MAP = {
    "departamento": "department", "codigodanedepartamento": "department_code",
    "municipio": "municipality", "codigodanemunicipio": "municipality_code",
    "ano": "year", "mes": "month", "tipodesminado": "demining_type",
    "areadespejada": "area_cleared_m2",
    "map": "items_mines", "muse": "items_uxo", "aei": "items_ieds",
    "totalartefactosdestruidos": "items_total", "organizacion": "operator",
    "estudiosnotecnicos": "non_technical_surveys", "estudiostecnicos": "technical_surveys",
    "operacionesdespeje": "clearance_operations",
    "areaspeligrosas": "hazardous_areas", "areaspeligrosasconfirmadas": "confirmed_hazardous_areas",
}
DEMINING_TYPES = {"Comunidades": "Communities", "Bases Militares": "Military bases"}
ITEMS = ["items_mines", "items_uxo", "items_ieds"]
RESULTS = ["area_cleared_m2", *ITEMS, "items_total",
           "non_technical_surveys", "technical_surveys", "clearance_operations"]
HAZARDS = ["hazardous_areas", "confirmed_hazardous_areas"]
MEASURES = RESULTS + HAZARDS


def period_of(year):
    """Return the analysis-plan period for a year: start-up, before or after 2016."""
    if year <= 2009:
        return START_UP
    return BEFORE if year <= 2016 else AFTER


def prepare(raw):
    """Return a renamed, typed copy of the raw table with operator type and period added."""
    df = raw.rename(columns=COLUMN_MAP)
    df["department_code"] = df["department_code"].astype(str).str.zfill(2)
    df["municipality_code"] = df["municipality_code"].astype(str).str.zfill(5)
    df["demining_type"] = df["demining_type"].map(DEMINING_TYPES)
    df["operator_type"] = np.where(df["operator"].isin(MILITARY_OPERATORS), "Military", "Civilian")
    df["period"] = df["year"].map(period_of)
    return df


df = prepare(df_raw)
assert df["demining_type"].notna().all(), "unexpected demining type"
df.head()

,department,department_code,municipality,municipality_code,year,month,demining_type,area_cleared_m2,items_mines,items_uxo,items_ieds,items_total,operator,non_technical_surveys,technical_surveys,clearance_operations,hazardous_areas,confirmed_hazardous_areas,operator_type,period
0,CAUCA,19,CALOTO,19142,2018,4,Communities,0.0,0,0,0,0,HI,3,0,0,0,0,Civilian,2017–2023
1,META,50,LA MACARENA,50350,2019,9,Communities,1563.0,0,148,0,148,BRDEH,0,0,1,0,0,Military,2017–2023
2,SANTANDER,68,SIMACOTA,68745,2020,7,Communities,6499.0,0,0,0,0,BRDEH,0,0,1,0,0,Military,2017–2023
3,TOLIMA,73,RONCESVALLES,73622,2022,10,Communities,0.0,0,0,0,0,BRDEH,0,0,0,1,0,Military,2017–2023
4,CAUCA,19,PÁEZ,19517,2023,2,Communities,0.0,0,0,0,0,HI,4,0,0,0,0,Civilian,2017–2023


### 2.3 Data checks

Before any totals are added up, four checks: empty cells and repeated rows; whether the item total equals its three parts; rows that share a municipality, month, demining type and operator; and which months have records.

In [5]:
print(f"Empty cells: {df.isna().sum().sum()}")
print(f"Rows that repeat another row exactly: {df.duplicated().sum()}")
items_match = df[ITEMS].sum(axis=1).eq(df["items_total"])
print(f"Rows where items_total equals mines + UXO + IEDs: {items_match.sum():,} of {len(df):,}")
print(f"Municipality codes: {df['municipality_code'].nunique()}; "
      f"municipality names: {df['municipality'].nunique()} (some names repeat across departments)")

Empty cells: 0
Rows that repeat another row exactly: 0
Rows where items_total equals mines + UXO + IEDs: 5,721 of 5,721
Municipality codes: 435; municipality names: 418 (some names repeat across departments)


In [6]:
KEY = ["municipality_code", "year", "month", "demining_type", "operator"]


def repeated_key_check(df, key):
    """Check rows that share a key: do they repeat a measure, or split the measures between them?

    Returns the number of key combinations that appear more than once, and for each measure the
    number of those combinations in which more than one row reports a non-zero value. A zero for
    every measure means the shared rows never repeat a measure, so summing rows double-counts nothing.
    """
    shared = df[df.duplicated(key, keep=False)]
    nonzero_rows = shared[MEASURES].ne(0).groupby([shared[k] for k in key]).sum()
    return len(nonzero_rows), (nonzero_rows > 1).sum()


combinations, repeated_measures = repeated_key_check(df, KEY)
print(f"Combinations of municipality, month, demining type and operator that appear twice: {combinations:,}")
print("Combinations in which a measure is non-zero in more than one row:")
print(repeated_measures.to_string())

reports_results = df[RESULTS].ne(0).any(axis=1)
reports_hazards = df[HAZARDS].ne(0).any(axis=1)
print("\nWhat each row reports:")
pd.crosstab(reports_results.map({True: "Reports results", False: "No results"}),
            reports_hazards.map({True: "Reports hazardous areas", False: "No hazardous areas"}),
            rownames=["Results"], colnames=["Hazardous areas"])

Combinations of municipality, month, demining type and operator that appear twice: 1,183
Combinations in which a measure is non-zero in more than one row:
area_cleared_m2              0
items_mines                  0
items_uxo                    0
items_ieds                   0
items_total                  0
non_technical_surveys        0
technical_surveys            0
clearance_operations         0
hazardous_areas              0
confirmed_hazardous_areas    0

What each row reports:


Hazardous areas,No hazardous areas,Reports hazardous areas
Results,,
No results,16,1705
Reports results,4000,0


In [7]:
records_per_month = df.groupby(["year", "month"]).size()
all_months = pd.MultiIndex.from_product([range(2004, 2024), range(1, 13)], names=["year", "month"])
missing = all_months.difference(records_per_month.index)
print(f"Months from 2004 to 2023 with no record: {len(missing)}")
print(pd.Series(missing.get_level_values("year")).value_counts().sort_index().to_string())

late = records_per_month.unstack("month").loc[[2022, 2023], 9:12].fillna(0).astype(int)
print("\nRecords in September to December, 2022 and 2023:")
late

Months from 2004 to 2023 with no record: 37
year
2004    11
2005    10
2006     5
2007     3
2008     4
2009     1
2011     1
2012     1
2023     1

Records in September to December, 2022 and 2023:


month,9,10,11,12
year,,,,
2022,56,54,47,25
2023,23,6,1,0


**What the checks show.**
- No cell is empty, no row repeats another exactly, and the item total always equals the three item types.
- 1,183 combinations of municipality, month, demining type and operator appear twice, but the two rows never report the same measure. No row reports both results (land cleared, items found, surveys, clearance operations) and hazardous areas, and each pair joins a row of hazardous areas with a row of results (in three pairs, with an empty row). Inference: the published file joins two kinds of record. Summing the rows therefore gives correct totals, and none of the paired rows should be deleted as duplicates.
- Of the 37 months with no record, 34 fall in the start-up years 2004 to 2009. Within the test window there are three: January 2011, January 2012 and December 2023.
- Records thin out at the end of 2023: 23 in September, 6 in October, 1 in November and none in December, against 56, 54, 47 and 25 in the same months of 2022. Inference: the rows were last updated on 19 January 2024, so the last months of 2023 were probably not yet fully reported. The plan counts December 2023 as zero; if that month is really missing, counting it as zero lowers the after-2016 group, so it works against finding a rise.
- The portal describes the data as starting in 2005; the file also holds one record from 2004, a military-base record with nothing cleared.